# Entraînement PPO du Go1 sur Kaggle

**Réglages du notebook (panneau de droite) :** Accelerator = **GPU T4 x2**, Internet = **On**.

Pour un entraînement long, ne pas garder l'onglet ouvert : **Save Version → Save & Run All (Commit)**.
Kaggle exécute alors tout en arrière-plan (jusqu'à 12 h) et garde `/kaggle/working` dans l'onglet *Output*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

## Installation
Le code est cloné depuis GitHub : il faut avoir poussé ses derniers commits.

In [ ]:
!rm -rf /tmp/rescue_dog
!git clone --depth 1 https://github.com/BOUBAKERAMDY/rlRobot3D.git /tmp/rescue_dog
%cd /tmp/rescue_dog
!git log --oneline -1
!pip install -q -e . "jax[cuda12]<0.10"

In [ ]:
import os

# Rendu sans écran pour les vidéos, et option XLA recommandée par MuJoCo Playground.
os.environ["MUJOCO_GL"] = "egl"
os.environ["XLA_FLAGS"] = "--xla_gpu_triton_gemm_any=True"

!python -c "import jax; print(jax.devices())"

## Entraînement
Les overrides remplacent les valeurs de `configs/base.yaml`. Avec la config de base (200 M pas),
compter quelques heures sur T4 ; pour un premier essai, `num_timesteps=50_000_000` suffit déjà à voir le chien marcher.

In [ ]:
RUN_NAME = "go1_baseline"
OVERRIDES = "num_timesteps=200_000_000"
RESULTS = "/kaggle/working/results"

!python scripts/train.py run_name={RUN_NAME} output_dir={RESULTS} {OVERRIDES}

## Courbe d'apprentissage

In [ ]:
import json
import matplotlib.pyplot as plt

run_dir = f"{RESULTS}/{RUN_NAME}"
rows = [json.loads(line) for line in open(f"{run_dir}/metrics.jsonl")]
steps = [r["step"] for r in rows]
reward = [r["eval/episode_reward"] for r in rows]
std = [r["eval/episode_reward_std"] for r in rows]

plt.plot(steps, reward)
plt.fill_between(steps, [m - s for m, s in zip(reward, std)], [m + s for m, s in zip(reward, std)], alpha=0.3)
plt.xlabel("pas d'environnement")
plt.ylabel("récompense d'évaluation")
plt.title(RUN_NAME)
plt.savefig(f"{run_dir}/reward.png", dpi=120)
plt.show()

## Vidéo
Commande joystick : (avant m/s, côté m/s, rotation rad/s).

In [ ]:
from IPython.display import Video

!python scripts/record_video.py {run_dir} --command 1.0 0 0
Video(f"{run_dir}/walk.mp4", embed=True, width=640)

## Téléchargement
Le zip apparaît dans l'onglet *Output* ; le décompresser dans `results/` sur le Mac.

In [ ]:
!cd {RESULTS} && zip -qr /kaggle/working/{RUN_NAME}.zip {RUN_NAME}
!ls -lh /kaggle/working/{RUN_NAME}.zip